# d2_teacher — Direction 2: label states with the oracle MPC teacher, for the student to imitate

`hvac_controller/d2/teacher.py` on the D1 replica caches (dataset `buildings`: 194 scenarios, `manifest_v2.json`).
Every hour the teacher (an MPC planning on the scenario's own exact model, with the true weather) labels the state
with its supervisory decisions; one `.npz` per scenario (features, decisions, hourly rewards).

- `MODE = "teacher"`: the teacher acts. `MODE = "dagger"`: a student acts (attach its `student.pt`), the teacher
  labels the states the student visits; `BETA` = share of hours the teacher's decision is executed instead.
- `TAGS`: `gen_train` (180 scenarios: 60 generated buildings x 3 climates). Shard over sessions with `RUN_ID`.

**Inputs:** the competition data, dataset `buildings` (and the student weights for `dagger`). CPU, **Internet on**.

**Output:** `/kaggle/working/labels_<MODE>_<RUN_ID>.zip` with the `.npz` files.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction2"
POLICY = "heavy.pa0.hs1"     # the teacher (an oracle preset)
MODE = "teacher"             # "teacher" or "dagger"
BETA = 0.3                   # dagger: share of hours the teacher acts
TAGS = "gen_train"
STEPS = 8640
RUN_ID, N_SHARDS = 0, 3
N_PROCS = 4

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")


def find(pattern):
    """The folder under /kaggle/input that holds a file matching pattern (datasets and notebook outputs alike)."""
    hits = sorted(INPUT.rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"no {pattern} under {INPUT}: attach the input")
    return hits[0].parent

CACHES = find("manifest_v2.json")
print("caches:", CACHES, len(list(CACHES.glob("*.pkl"))), "pkl")
STUDENT = None
if MODE == "dagger":
    STUDENT = next(INPUT.rglob("student.pt"))
    print("student:", STUDENT)

In [ ]:
OUT = WORK / "labels"
cmd = [sys.executable, "-m", "hvac_controller.d2.teacher", str(OUT), "--caches", str(CACHES),
       "--manifest", str(CACHES / "manifest_v2.json"), "--tags", TAGS, "--policy", POLICY, "--mode", MODE,
       "--beta", str(BETA), "--steps", str(STEPS), "--shard", str(RUN_ID), "--shards", str(N_SHARDS),
       "--n-procs", str(N_PROCS)] + (["--student", str(STUDENT)] if STUDENT else [])
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith("[") or "Error" in line or "Traceback" in line:
        print(line, end="", flush=True)
print("exit", proc.wait(), f"after {(time.time() - t0) / 60:.0f} min")

In [ ]:
import numpy as np

files = sorted(OUT.glob("*.npz"))
scores = [float(np.load(p)["score"]) for p in files]
print(f"{len(files)} scenarios labelled, mean teacher score {np.mean(scores) if scores else float('nan'):.5f}")
name = Path(f"/kaggle/working/labels_{MODE}_{RUN_ID}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.name)
print("wrote", name)